# Week 5 Deep Learning Application: Vehicle Valuation Neural Network

**Dataset:** CarDekho Car Market Trends Data  
**Task:** Supervised Deep Learning Regression (Predict Selling Price)  
**Framework:** TensorFlow / Keras

In [ ]:
# ==============================================================================
# WEEK 5 DEEP LEARNING APPLICATION: VEHICLE VALUATION NEURAL NETWORK
# Dataset: CarDekho Car Market Trends Data
# Task: Supervised Deep Learning Regression (Predict Selling Price)
# Framework: TensorFlow / Keras
# ==============================================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout, Input
from tensorflow.keras.models import Sequential

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# ------------------------------------------------------------------------------
# 1. DATA LOADING & FEATURE ENGINEERING
# ------------------------------------------------------------------------------
df = pd.read_csv("1776311302-P3-Car Market Trends Analysis with Car Dekho Data.csv")

# Engineer Vehicle Age feature
CURRENT_YEAR = 2026
df["Age"] = CURRENT_YEAR - df["Year"]

# Define Features (X) and Target (y)
feature_cols = [
    "Present_Price",
    "Kms_Driven",
    "Age",
    "Owner",
    "Fuel_Type",
    "Seller_Type",
    "Transmission",
]
X = df[feature_cols]
y = df["Selling_Price"]

# One-Hot Encoding for categorical features
X_encoded = pd.get_dummies(
    X, columns=["Fuel_Type", "Seller_Type", "Transmission"], drop_first=True
)

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42
)

# Standard Z-Score Normalization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(
    f"Training set size: {X_train_scaled.shape[0]} samples, {X_train_scaled.shape[1]} features"
)
print(f"Test set size:     {X_test_scaled.shape[0]} samples")

# ------------------------------------------------------------------------------
# 2. NEURAL NETWORK ARCHITECTURE DESIGN
# ------------------------------------------------------------------------------
model = Sequential(
    [
        Input(shape=(X_train_scaled.shape[1],)),
        # Layer 1: Dense Expansion + Batch Normalization + Dropout
        Dense(64, activation="relu", name="Input_Expansion_Layer"),
        BatchNormalization(),
        Dropout(0.1, name="Dropout_1"),
        # Layer 2: Feature Representation
        Dense(32, activation="relu", name="Representation_Layer"),
        BatchNormalization(),
        # Layer 3: Bottleneck Compression Layer
        Dense(16, activation="relu", name="Bottleneck_Layer"),
        # Output Layer: Single linear output unit for regression
        Dense(1, activation="linear", name="Output_Layer"),
    ],
    name="CarDekho_Valuation_ANN",
)

# Display Architecture
model.summary()

# ------------------------------------------------------------------------------
# 3. MODEL COMPILATION & TRAINING
# ------------------------------------------------------------------------------
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="mse",
    metrics=["mae"],
)

# Callbacks to prevent overfitting and adjust learning rate dynamically
early_stopping = EarlyStopping(
    monitor="val_loss", patience=25, restore_best_weights=True, verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=10, min_lr=0.0001, verbose=1
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=200,
    batch_size=16,
    callbacks=[early_stopping, reduce_lr],
    verbose=0,
)

# ------------------------------------------------------------------------------
# 4. EVALUATION & METRICS
# ------------------------------------------------------------------------------
y_pred = model.predict(X_test_scaled).flatten()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n" + "=" * 45)
print("     DEEP LEARNING MODEL EVALUATION METRICS     ")
print("=" * 45)
print(f"Mean Absolute Error (MAE)  : {mae:.4f} Lakhs")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f} Lakhs")
print(f"Coefficient of Determination (R²): {r2:.4f}")
print("=" * 45)

# ------------------------------------------------------------------------------
# 5. TRAINING VISUALIZATIONS
# ------------------------------------------------------------------------------
plt.figure(figsize=(12, 4))

# Loss Curve
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train Loss (MSE)", color="#1E3A8A")
plt.plot(history.history["val_loss"], label="Val Loss (MSE)", color="#DC2626")
plt.title("Neural Network Loss Convergence")
plt.xlabel("Epochs")
plt.ylabel("Mean Squared Error")
plt.legend()

# MAE Curve
plt.subplot(1, 2, 2)
plt.plot(history.history["mae"], label="Train MAE", color="#1E3A8A")
plt.plot(history.history["val_mae"], label="Val MAE", color="#DC2626")
plt.title("Mean Absolute Error Across Epochs")
plt.xlabel("Epochs")
plt.ylabel("MAE (in Lakhs)")
plt.legend()

plt.tight_layout()
plt.show()
